<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-09-ai-agents/lesson-9.3-langgraph-deep-dive/notebooks/exercises-9.3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 9.3 — LangGraph Deep Dive
Netsetos GenAI Course v2.0 | Module 9: AI Agents

Reducers, prebuilt, parallel/Send, interrupt/HITL, checkpointing, streaming, RetryPolicy, Platform, LangSmith, India


In [ ]:
print('Module 9: AI Agents')
print('Lesson 9.3: LangGraph Deep Dive')


## Ex 1: Reducer Types


In [ ]:
print('LangGraph Reducer Types:')
for rtype, syntax, behavior in [
    ('None (default)','current_step: str','Last writer wins (overwrite)'),
    ('operator.add','results: Annotated[list, add]','Concatenate lists / sum ints'),
    ('add_messages','messages: Annotated[list, add_messages]','Smart append + ID dedup'),
    ('Custom lambda','meta: Annotated[dict, lambda c,n: {**c,**n}]','Merge dicts'),
]: print(f'  {rtype:18s}: {syntax}')
   print(f'  {"":18s}  → {behavior}')


## Ex 2: create_react_agent internals


In [ ]:
print('create_react_agent builds this graph:')
print()
print('  START → [agent] → tools_condition → [tools] → [agent] ...')
print('                  ↘ END (no tool calls)')
print()
print('Equivalent manual construction:')
print('  builder = StateGraph(MessagesState)')
print('  builder.add_node("agent", llm_with_tools)')
print('  builder.add_node("tools", ToolNode(tools))')
print('  builder.add_edge(START, "agent")')
print('  builder.add_conditional_edges("agent", tools_condition)')
print('  builder.add_edge("tools", "agent")')


## Ex 3: Send API Map-Reduce


In [ ]:
print('Map-Reduce with Send API:')
print()
print('def fan_out(state):')
print('    return [Send("analyze", {"doc": d}) for d in state["docs"]]')
print()
print('# Each Send creates INDEPENDENT parallel instance')
print('# Reducer aggregates: results: Annotated[list, operator.add]')
print()
for n_docs in [3,5,10,20]:
    print(f'  {n_docs} docs → {n_docs} parallel instances → 1 aggregated result')


## Ex 4: interrupt() Rules


In [ ]:
print('interrupt() Critical Rules:')
for i,rule in enumerate([
    'Node RESTARTS from beginning on resume',
    'Pre-interrupt side effects must be IDEMPOTENT',
    'NEVER wrap interrupt() in try/except',
    'Do NOT reorder interrupt() calls (matched by index)',
    'Requires checkpointer + thread_id',
    'Resume: Command(resume=value) → becomes return of interrupt()',
],1): print(f'  {i}. {rule}')


## Ex 5: Checkpoint Backend Selection


In [ ]:
print('Backend selection guide:')
for be, pkg, use in [
    ('InMemorySaver','Built-in','Dev/testing ONLY'),
    ('SqliteSaver','langgraph-checkpoint-sqlite','Local dev, notebooks'),
    ('PostgresSaver','langgraph-checkpoint-postgres','Production default (ACID)'),
    ('RedisSaver','langgraph-checkpoint-redis','High-throughput + TTL'),
    ('DynamoDBSaver','langgraph-checkpoint-aws','AWS serverless'),
]: print(f'  {be:16s}: {use}')
   print(f'  {"":16s}  pip install {pkg}')


## Ex 6: 6 Streaming Modes


In [ ]:
print('LangGraph streaming modes:')
for mode, returns, use in [
    ('updates','State delta per node','Production UIs (default)'),
    ('values','Full state per superstep','Debugging dashboards'),
    ('messages','(chunk, metadata) tuples','Chat UIs (token-by-token)'),
    ('custom','User-defined via StreamWriter','Progress bars'),
    ('events','Lifecycle events','LCEL migration'),
    ('debug','Full trace before/after','Development only'),
]: print(f'  {mode:10s}: {returns:35s} | {use}')


---
## Recap
Reducers (add_messages > operator.add > custom). create_react_agent = agent→tools_condition→ToolNode. Send API for dynamic map-reduce. interrupt() for HITL (node restarts on resume!). 5 checkpoint backends (Postgres=default, Redis=speed+TTL). 6 stream modes (updates=production, messages=chat). RetryPolicy per node. LangGraph Platform (CLI→Studio→Cloud). LangSmith auto-tracing. Sarvam OpenAI-compatible for Indian languages.
